In [2]:
from langgraph.graph import START,END,StateGraph
from typing import Annotated,TypedDict,Literal
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import AIMessage,HumanMessage,ToolMessage
import os
from dotenv import load_dotenv
from langgraph.graph.message import add_messages

load_dotenv(dotenv_path=".env")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not found"
    )

In [5]:

class AgentState(TypedDict):
    messages: Annotated[list,add_messages]

from langgraph.graph.message import add_messages as add_msg_fn

existing=[HumanMessage(content="Hello !")]

new=[AIMessage(content="Hi there")]

combined=add_msg_fn(existing,new)

for msg in combined:
    print(f"{type(msg).__name__}  : {msg.content}")

HumanMessage  : Hello !
AIMessage  : Hi there


In [8]:
@tool
def get_weather(city:str)->str:
    """Get the current weather for a given city.
    
    Args:
        city: The name of the city to get weather for.
        
    Returns:
        A string describing the current weather.
    """
    weather_data = {
        "paris": "Sunny, 22°C",
        "london": "Cloudy, 15°C",
        "tokyo": "Rainy, 18°C",
        "new york": "Partly cloudy, 20°C",
    }

    city_lower=city.lower()
   
    if city_lower in weather_data: 
        return f"The weather for {city} is {weather_data[city_lower]}"
    else:
        return f"{city} not found"

print("Testing Tool:")
print(get_weather.invoke({"city":"Paris"}))
    

Testing Tool:
The weather for Paris is Sunny, 22°C


In [10]:
llm=ChatOpenAI(model='gpt-4o-mini',temperature=0.7)
tools=[get_weather]
llm_with_tools=llm.bind_tools(tools)

print(f"LLM configured with tools")

LLM configured with tools


In [ ]:
from typing import Literal

class AgentState(TypedDict):
    messages:Annotated[list,add_messages]

def call_model(state:AgentState):
    messages=state['messages']
    response=llm_with_tools.invoke(messages)
    return {"messages":[response]}

def tool_calls(state:AgentState):
    message=state['messages']
    ai_message=message[-1]
    tool_calls=ai_message.tool_calls

    tool_messages=[]
    for tool_call in tool_calls:
        tool_name=tool_call["name"]
        tool_args=tool_call["args"]

        print(f"Calling tool: {tool_name}:{tool_args}")
        tool={t.name:t for t in tools}[tool_name]
        result=tool.invoke(tool_args)

        tool_messages.append(ToolMessage(
            content=result,
            tool_call_id=tool_call["id"]
        ))

        return {"message":tool_messages}
